# TP4 - Transformers in Action : directions futures et applications multimodales
**Rapport associé :** `Rapport_Transformers.pdf` - ES-SAOUDY Zakaria

Questions du rapport : jusqu'où scaler, comment dépasser le texte, et comment aligner les modalités.

In [ ]:
%pip install -q transformers torch pillow pandas matplotlib datasets soundfile

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt, torch
from PIL import Image
from transformers import pipeline, CLIPProcessor, CLIPModel

## 1. Architectures étudiées
- **ViT** : patches projetés en tokens, token de classification, encodeur Transformer.
- **iGPT** : pixels quantifiés comme tokens et prédiction autorégressive du pixel suivant.
- **TAPAS** : BERT enrichi par positions ligne/colonne et agrégateurs NONE/SUM/COUNT/AVERAGE.
- **wav2vec 2.0** : convolutions sur audio brut, masquage latent et apprentissage contrastif.
- **CLIP** : alignement contrastif de 400 M de paires image-texte, classification zero-shot.

## 2. Croissance des modèles et échelle logarithmique

In [ ]:
model_data = [
    {'date':'12-06-2017','name':'Transformer','size':213e6},
    {'date':'11-06-2018','name':'GPT','size':110e6},
    {'date':'11-10-2018','name':'BERT','size':340e6},
    {'date':'14-02-2019','name':'GPT-2','size':1.5e9},
    {'date':'23-10-2019','name':'T5','size':11e9},
    {'date':'17-09-2019','name':'Megatron','size':8.3e9},
    {'date':'13-02-2020','name':'Turing-NLG','size':17e9},
    {'date':'30-06-2020','name':'GShard','size':600e9},
    {'date':'28-05-2020','name':'GPT-3','size':175e9},
    # 1.6 T selon le tableau et le facteur x7500 du rapport (coquille e13 corrigée dans l'annexe).
    {'date':'11-01-2021','name':'Switch-C','size':1.6e12},
]
scale = pd.DataFrame.from_records(model_data); scale['date'] = pd.to_datetime(scale['date'], dayfirst=True)
fig, ax = plt.subplots(figsize=(12,4)); scale.plot(x='date',y='size',kind='scatter',s=15,ax=ax)
for row in scale.itertuples(): ax.annotate(row.name,(row.date,row.size),xytext=(3,3),textcoords='offset points',fontsize=8)
ax.set_yscale('log'); ax.set_xlabel('Release date'); ax.set_ylabel('Number of parameters'); ax.grid(True)
plt.tight_layout(); plt.show()

## 3. Lois d'échelle et défis
Kaplan et al. donnent des lois de puissance pour paramètres `N`, données `D` et calcul `C`, avec
`alpha_N ≈ 0.076` et `alpha_C ≈ 0.050`. Défis : attention quadratique `O(n²)`, coût carbone, biais et
sécurité. Réponses : Sparse Attention (Longformer/BigBird), attention linéarisée et efficience.

## 4. ViT - classification d'images, fallback rouge 100x100 exact

In [ ]:
try:
    image = Image.open('images/doge.jpg')
except FileNotFoundError:
    image = Image.new('RGB', (100,100), color='red')
image_classifier = pipeline('image-classification', model='google/vit-base-patch16-224')
vit_preds = image_classifier(image)
print(pd.DataFrame(vit_preds))

## 5. TAPAS - table et quatre requêtes du rapport

In [ ]:
book_data = [
    {'chapter':0,'name':'Introduction','start_page':1,'end_page':11},
    {'chapter':1,'name':'Text classification','start_page':12,'end_page':48},
    {'chapter':2,'name':'Named Entity Recognition','start_page':49,'end_page':73},
    {'chapter':3,'name':'Question Answering','start_page':74,'end_page':120},
    {'chapter':4,'name':'Summarization','start_page':121,'end_page':140},
    {'chapter':5,'name':'Conclusion','start_page':141,'end_page':144},
]
table = pd.DataFrame(book_data); table['number_of_pages'] = table.end_page - table.start_page
table = table.astype(str)
table_qa = pipeline('table-question-answering', model='google/tapas-base-finetuned-wtq')
queries = ["What's the topic in chapter 4?", "What is the total number of pages?",
           "On which page does the chapter about question-answering start?",
           "How many chapters have more than 20 pages?"]
tapas_preds = table_qa(table, queries)
for query, pred in zip(queries, tapas_preds): print(query, '=>', pred['answer'])

Résultats rapportés : `Summarization`, `SUM > 10,36,24,46,19,3` (=138), `AVERAGE > 74`,
`COUNT > 1,2,3`. Le COUNT retourne les indices éligibles, limite connue du modèle base.

## 6. wav2vec 2.0 - pipeline et échantillon SUPERB exacts

In [ ]:
from datasets import load_dataset
import soundfile as sf
asr = pipeline('automatic-speech-recognition', model='facebook/wav2vec2-base-960h')
try:
    ds = load_dataset('superb','asr',split='validation[:1]')
    def map_to_array(batch):
        speech, _ = sf.read(batch['file']); batch['speech'] = speech; return batch
    ds = ds.map(map_to_array); ds.set_format('numpy')
    asr_pred = asr(ds[0]['speech']); print(asr_pred)
except Exception as error:
    print('Erreur chargement dataset décrite dans le rapport :', error)

Le rapport observe localement une erreur d'URI Hugging Face liée à la version de `datasets`, alors que le
pipeline `facebook/wav2vec2-base-960h` s'initialise. La transcription attendue commence par
`MISTER QUILTER IS THE APOSTLE OF THE MIDDLE...` lorsque SUPERB est accessible.

## 7. CLIP - image `images/optimusprime.jpg` et trois textes exacts

In [ ]:
# Dans Colab, téléverser l'image du rapport si elle n'est pas déjà présente.
clip_path = Path('images/optimusprime.jpg')
if not clip_path.exists():
    clip_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
        print("Téléversez l'image optimusprime.jpg utilisée dans le rapport.")
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded)); clip_path.write_bytes(uploaded[uploaded_name])
    except Exception:
        raise FileNotFoundError('Ajoutez images/optimusprime.jpg avant cette cellule.')

clip_ckpt = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(clip_ckpt)
processor = CLIPProcessor.from_pretrained(clip_ckpt)
clip_image = Image.open(clip_path)
texts = ['a photo of a transformer','a photo of a robot','a photo of agi']
inputs = processor(text=texts, images=clip_image, return_tensors='pt', padding=True)
with torch.no_grad(): outputs = clip_model(**inputs)
probs = outputs.logits_per_image.softmax(dim=1)
for text, prob in zip(texts, probs[0]): print(f'{text:35s} -> {prob.item():.4f}')

## 8. Perspectives et directions ouvertes
- **DALL-E** : génération image-texte autorégressive.
- **wav2vec-U** : ASR sans transcription par alignement adversarial, utile aux langues peu dotées.
- **Mixture of Experts / Switch-C** : activation sparse d'experts, calcul décorrélé du nombre total de poids.

## 9. Synthèse, améliorations et consolidation
La scalabilité est prévisible mais coûteuse ; l'attention efficiente ouvre les longues séquences ; CLIP montre
la puissance de l'alignement contrastif. Les erreurs de versions et médias absents démontrent que la
reproductibilité compte autant que l'algorithme. Consolidations proposées par le rapport : versions fixées,
médias fournis, attention sparse/linéaire, meilleure efficience, audit des biais et validation multimodale.